In [5]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.applications import ResNet50V2,MobileNetV2
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, GlobalAveragePooling2D, 
    Input, LayerNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
from tqdm import tqdm

In [6]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [7]:
class VideoClassifier:
    
    def __init__(self, max_frames=30, frame_size=(224, 224)):
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.feature_extractor = self._build_feature_extractor()
        self.model = None
        logger.info(f"Initialized VideoClassifier with max_frames={max_frames}, frame_size={frame_size}")
        
    def _build_feature_extractor(self):
        base_model = MobileNetV2(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.frame_size, 3)
        )
        model = Model(
            inputs=base_model.input,
            outputs=GlobalAveragePooling2D()(base_model.output)
        )
        return model
    
    def extract_frames(self, video_path):
        try:
            if not str(video_path).endswith('.mp4'):
                video_path = str(video_path) + '.mp4'
            
            if not os.path.exists(video_path):
                logger.warning(f"Video file not found: {video_path}")
                return None
                
            frames = []
            cap = cv2.VideoCapture(video_path)
            
            if not cap.isOpened():
                logger.warning(f"Failed to open video: {video_path}")
                return None

            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames == 0:
                logger.warning(f"Video has no frames: {video_path}")
                return None
            #####
            indices = np.linspace(0, total_frames - 1, self.max_frames, dtype=int)
            
            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if ret:
                    frame = cv2.resize(frame, self.frame_size)
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = frame.astype(np.float32) / 255.0
                    frames.append(frame)
                else:
                    frames.append(np.zeros((*self.frame_size, 3)))
            
            cap.release()
            
            if len(frames) < self.max_frames:
                logger.warning(f"Not enough frames in video: {video_path}")
                return None
                
            return np.array(frames)
            ##### 
            
        except Exception as e:
            logger.error(f"Error processing video {video_path}: {str(e)}")
            if 'cap' in locals():
                cap.release()
            return None

    def prepare_data(self, video_dir, metadata_path):
        
        try:
            video_dir = Path(video_dir)
            metadata_path = Path(metadata_path)
            
            if not video_dir.exists():
                raise ValueError(f"Video directory not found: {video_dir}")
            if not metadata_path.exists():
                raise ValueError(f"Metadata file not found: {metadata_path}")
                
            logger.info(f"Reading metadata from: {metadata_path}")
            metadata = pd.read_excel(metadata_path)
            logger.info(f"Original metadata shape: {metadata.shape}")
            
            if 'VideoID' not in metadata.columns or 'HowFunny' not in metadata.columns:
                raise ValueError("Required columns 'VideoID' or 'HowFunny' not found in metadata")
            
            logger.info("\nOriginal HowFunny values:\n%s", metadata['HowFunny'].value_counts())
            
            def combine_categories(x):
                if pd.isna(x):
                    return None
                x = str(x).strip()
                if x in ['Funny', 'Very funny', 'Extremely funny']:
                    return 'Very Funny'
                elif x == 'Slightly funny':
                    return 'Slightly Funny'
                elif x == 'Not funny':
                    return 'Not Funny'
                else:
                    return None

            metadata['HowFunny_Combined'] = metadata['HowFunny'].apply(combine_categories)
            metadata = metadata.dropna(subset=['HowFunny_Combined'])
            
            category_mapping = {
                'Not Funny': 0,
                'Slightly Funny': 1,
                'Very Funny': 2
            }
            metadata['Label'] = metadata['HowFunny_Combined'].map(category_mapping)
            
            logger.info("\nCombined category distribution:\n%s", metadata['HowFunny_Combined'].value_counts())
            logger.info("\nNumerical label distribution:\n%s", metadata['Label'].value_counts())
            
            videos_list = []
            labels = []
            processed_videos = []
            
            logger.info(f"\nProcessing {len(metadata)} videos...")
            
            for idx, row in tqdm(metadata.iterrows(), total=len(metadata)):
                try:
                    video_id = str(row['VideoID']).strip()
                    video_path = video_dir / video_id
                    
                    frames = self.extract_frames(video_path)
                    if frames is not None:
                        
                        # Features extraction ffor each frame
                        video_features = []
                        for frame in frames:
                            frame_features = self.feature_extractor.predict(
                                np.expand_dims(frame, axis=0),
                                verbose=0
                            )
                            video_features.append(frame_features[0])
                        
                        videos_list.append(video_features)
                        labels.append(int(row['Label']))
                        processed_videos.append(video_id)
                        
                        if len(videos_list) % 10 == 0:
                            logger.info(f"Successfully processed {len(videos_list)} videos")
                            
                except Exception as e:
                    logger.error(f"Error processing video {row['VideoID']}: {str(e)}")
                    continue
            
            if not videos_list:
                logger.error("\nNo videos were successfully processed.")
                raise ValueError("No valid videos processed")
                
            X = np.array(videos_list, dtype=np.float32)
            y = np.array(labels, dtype=np.int32)
            
            logger.info(f"\nFinal dataset:")
            logger.info(f"Features shape: {X.shape}")
            logger.info(f"Labels shape: {y.shape}")
            logger.info(f"Label distribution: {np.unique(y, return_counts=True)}")
            
            output_dir = Path('outputs')
            output_dir.mkdir(exist_ok=True)
            
            with open(output_dir / 'processed_videos.txt', 'w') as f:
                for vid in processed_videos:
                    f.write(f"{vid}\n")
            
            return X, y
            
        except Exception as e:
            logger.error(f"\nError preparing dataset: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            raise

    def build_model(self, input_shape, num_classes):
        inputs = Input(shape=input_shape)
        
        x = LSTM(256, return_sequences=True)(inputs)
        x = LayerNormalization()(x)
        x = Dropout(0.2)(x)
        
        
        x = LSTM(256)(x)
        x = LayerNormalization()(x)
        x = Dropout(0.2)(x)
                
        
        x = Dense(128, activation='relu')(x)
        outputs = Dense(num_classes, activation='softmax')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        model.compile(
            optimizer=SGD(1e-3),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model
    
    def train(self, video_dir, metadata_path, epochs=50, batch_size=16):
        X, y = self.prepare_data(video_dir, metadata_path)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        input_shape = (X.shape[1], X.shape[2])  # (timesteps, features)
        
        self.model = self.build_model(
            input_shape=input_shape,
            num_classes=len(np.unique(y))
        )
        
        output_dir = Path('outputs')
        output_dir.mkdir(exist_ok=True)
        
        callbacks = [
            EarlyStopping(patience=10, restore_best_weights=True),
            ModelCheckpoint(
                output_dir / 'best_model.keras',
                save_best_only=True
            )
        ]
        
        history = self.model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks
        )
        
        self.evaluate(X_test, y_test, history)
        
    def evaluate(self, X_test, y_test, history):
        
        output_dir = Path('outputs')
        output_dir.mkdir(exist_ok=True)
        
        y_pred = np.argmax(self.model.predict(X_test), axis=1)
        
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(history.history['accuracy'], label='Train')
        plt.plot(history.history['val_accuracy'], label='Validation')
        plt.title('Model Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        plt.plot(history.history['loss'], label='Train')
        plt.plot(history.history['val_loss'], label='Validation')
        plt.title('Model Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig(output_dir / 'training_history.png')
        plt.close()
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(
            confusion_matrix(y_test, y_pred),
            annot=True,
            fmt='d',
            cmap='Blues'
        )
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.savefig(output_dir / 'confusion_matrix.png')
        plt.close()
        
        report = classification_report(y_test, y_pred)
        logger.info("\nClassification Report:\n%s", report)
        
        with open(output_dir / 'classification_report.txt', 'w') as f:
            f.write(report)



In [8]:
try:
    video_dir = '/Users/jeremiah/phd_thesis/data/main_data/videos'
    metadata_path = '/Users/jeremiah/phd_thesis/data/main_data/excel2.xlsx'
    
    logger.info(f"Video directory: {video_dir}")
    logger.info(f"Metadata path: {metadata_path}")
    
    classifier = VideoClassifier(max_frames=10, frame_size=(224, 224))
    classifier.train(
        video_dir=video_dir,
        metadata_path=metadata_path,
        epochs=50,
        batch_size=16
    )
except Exception as e:
    logger.error(f"Error in main: {str(e)}")
    import traceback
    logger.error(traceback.format_exc())     



2024-12-17 21:48:11,979 - INFO - Video directory: /Users/jeremiah/phd_thesis/data/main_data/videos
2024-12-17 21:48:11,980 - INFO - Metadata path: /Users/jeremiah/phd_thesis/data/main_data/excel2.xlsx
2024-12-17 21:48:12,297 - INFO - Initialized VideoClassifier with max_frames=10, frame_size=(224, 224)
2024-12-17 21:48:12,298 - INFO - Reading metadata from: /Users/jeremiah/phd_thesis/data/main_data/excel2.xlsx
2024-12-17 21:48:12,499 - INFO - Original metadata shape: (1426, 23)
2024-12-17 21:48:12,503 - INFO - 
Original HowFunny values:
HowFunny
Not funny          549
Slightly funny     528
Funny              274
Very funny          68
Extremely funny      5
Name: count, dtype: int64
2024-12-17 21:48:12,510 - INFO - 
Combined category distribution:
HowFunny_Combined
Not Funny         549
Slightly Funny    528
Very Funny        347
Name: count, dtype: int64
2024-12-17 21:48:12,510 - INFO - 
Numerical label distribution:
Label
0    549
1    528
2    347
Name: count, dtype: int64
2024-12-

Epoch 1/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.3333 - loss: 1.3111 - val_accuracy: 0.3816 - val_loss: 1.1212
Epoch 2/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.3914 - loss: 1.1229 - val_accuracy: 0.4079 - val_loss: 1.0882
Epoch 3/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.4617 - loss: 1.0670 - val_accuracy: 0.4386 - val_loss: 1.0673
Epoch 4/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5002 - loss: 1.0441 - val_accuracy: 0.4430 - val_loss: 1.0343
Epoch 5/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.5198 - loss: 0.9752 - val_accuracy: 0.4342 - val_loss: 1.0493
Epoch 6/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5136 - loss: 0.9787 - val_accuracy: 0.4386 - val_loss: 1.0368
Epoch 7/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.5458 - loss: 0.9239 - val_accuracy: 0.4868 - val_loss: 1.0288
Epoch 8/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5540 - loss: 0.9015 - val_accuracy: 0.4956 - v

2024-12-17 21:57:46,869 - INFO - 
Classification Report:
              precision    recall  f1-score   support

           0       0.54      0.49      0.51       110
           1       0.44      0.45      0.45       106
           2       0.40      0.45      0.42        69

    accuracy                           0.47       285
   macro avg       0.46      0.46      0.46       285
weighted avg       0.47      0.47      0.47       285



# With Data Augumentation

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, GlobalAveragePooling2D, 
    Input, LayerNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam,SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
from tqdm import tqdm
import random

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [17]:
class VideoClassifier:
    
    def __init__(self, max_frames=30, frame_size=(224, 224)):
        
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.feature_extractor = self._build_feature_extractor()
        self.model = None
        logger.info(f"Initialized VideoClassifier with max_frames={max_frames}, frame_size={frame_size}")
        
    def _build_feature_extractor(self):
            base_model = ResNet50V2(
                weights='imagenet',
                include_top=False,
                input_shape=(*self.frame_size, 3)
            )
            model = Model(
                inputs=base_model.input,
                outputs=GlobalAveragePooling2D()(base_model.output)
            )
            return model
        
    def augment_frame(self, frame):
        
        """Apply random augmentations to a single frame"""
        
        try:
            
            # Convert to tensor
            frame = tf.convert_to_tensor(frame)
            frame = tf.expand_dims(frame, 0)  # Add batch dimension
            
            # Random horizontal flip
            if random.random() > 0.5:
                frame = tf.image.random_flip_left_right(frame)
            
            # Random brightness
            if random.random() > 0.5:
                frame = tf.image.random_brightness(frame, 0.2)
            
            # Random contrast
            if random.random() > 0.5:
                frame = tf.image.random_contrast(frame, 0.8, 1.2)
            
            # Random saturation
            if random.random() > 0.5:
                frame = tf.image.random_saturation(frame, 0.8, 1.2)
            
            # Random hue
            if random.random() > 0.5:
                frame = tf.image.random_hue(frame, 0.1)
                
            # Convert back to numpy and ensure proper range
            frame = tf.clip_by_value(frame, 0, 1)
            frame = frame[0].numpy()  # Remove batch dimension
            
            return frame
            
        except Exception as e:
            logger.error(f"Error in frame augmentation: {str(e)}")
            return frame
        
    def extract_frames(self, video_path, augment=True):
        
        """Extract frames from video with optional augmentation"""
        try:
            if not str(video_path).endswith('.mp4'):
                video_path = str(video_path) + '.mp4'
            
            if not os.path.exists(video_path):
                logger.warning(f"Video file not found: {video_path}")
                return None
                
            frames = []
            cap = cv2.VideoCapture(video_path)
            
            if not cap.isOpened():
                logger.warning(f"Failed to open video: {video_path}")
                return None
    
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames == 0:
                logger.warning(f"Video has no frames: {video_path}")
                return None
    
            # Dynamic sampling logic
            
            sample_interval = max(1, total_frames // self.max_frames)
            frame_count = 0
            sampled_count = 0
            
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                
                if frame_count % sample_interval == 0:
                    frame = cv2.resize(frame, self.frame_size)
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = frame.astype(np.float32) / 255.0
                    
                    if augment:
                        frame = self.augment_frame(frame)
                        
                    frames.append(frame)
                    sampled_count += 1
                
                frame_count += 1
            
            cap.release()
            
            while len(frames) < self.max_frames:
                frames.append(np.zeros((*self.frame_size, 3)))
            
            return np.array(frames[:self.max_frames])
            
        except Exception as e:
            logger.error(f"Error processing video {video_path}: {str(e)}")
            if 'cap' in locals():
                cap.release()
            return None
        
    def prepare_data(self, video_dir, metadata_path):
        
        try:
            video_dir = Path(video_dir)
            metadata_path = Path(metadata_path)
            
            if not video_dir.exists():
                raise ValueError(f"Video directory not found: {video_dir}")
            if not metadata_path.exists():
                raise ValueError(f"Metadata file not found: {metadata_path}")
                
            logger.info(f"Reading metadata from: {metadata_path}")
            metadata = pd.read_excel(metadata_path)
            logger.info(f"Original metadata shape: {metadata.shape}")
            
            if 'VideoID' not in metadata.columns or 'HowFunny' not in metadata.columns:
                raise ValueError("Required columns 'VideoID' or 'HowFunny' not found in metadata")
            
            logger.info("\nOriginal HowFunny values:\n%s", metadata['HowFunny'].value_counts())
    
            def combine_categories(x):
                if pd.isna(x):
                    return None
                x = str(x).strip()
                if x in ['Funny', 'Very funny', 'Extremely funny']:
                    return 'Very Funny'
                elif x == 'Slightly funny':
                    return 'Slightly Funny'
                elif x == 'Not funny':
                    return 'Not Funny'
                else:
                    return None

            metadata['HowFunny_Combined'] = metadata['HowFunny'].apply(combine_categories)
            metadata = metadata.dropna(subset=['HowFunny_Combined'])
            
            category_mapping = {
                'Not Funny': 0,
                'Slightly Funny': 1,
                'Very Funny': 2
            }
            metadata['Label'] = metadata['HowFunny_Combined'].map(category_mapping)
            
            logger.info("\nCombined category distribution:\n%s", metadata['HowFunny_Combined'].value_counts())
            logger.info("\nNumerical label distribution:\n%s", metadata['Label'].value_counts())
            
            videos_list = []
            labels = []
            processed_videos = []
            
            logger.info(f"\nProcessing {len(metadata)} videos...")
            
            for idx, row in tqdm(metadata.iterrows(), total=len(metadata), desc="Processing videos"):
                try:
                    video_id = str(row['VideoID']).strip()
                    video_path = video_dir / video_id
                    
                    frames = self.extract_frames(video_path, augment=False)
                    if frames is not None:
                        
                        video_features = []
                        for frame in frames:
                            frame_features = self.feature_extractor.predict(
                                np.expand_dims(frame, axis=0),
                                verbose=0
                            )
                            video_features.append(frame_features[0])
                        
                        videos_list.append(video_features)
                        labels.append(int(row['Label']))
                        processed_videos.append(video_id)
                        
                        # Add augmented version for training data
                        augmented_frames = self.extract_frames(video_path, augment=True)
                        if augmented_frames is not None:
                            aug_video_features = []
                            for frame in augmented_frames:
                                frame_features = self.feature_extractor.predict(
                                    np.expand_dims(frame, axis=0),
                                    verbose=0
                                )
                                aug_video_features.append(frame_features[0])
                            
                            videos_list.append(aug_video_features)
                            labels.append(int(row['Label']))
                            processed_videos.append(video_id + "_augmented")
                            
                except Exception as e:
                    logger.error(f"Error processing video {row['VideoID']}: {str(e)}")
                    continue
            
            if not videos_list:
                logger.error("\nNo videos were successfully processed.")
                raise ValueError("No valid videos processed")
                
            X = np.array(videos_list, dtype=np.float32)
            y = np.array(labels, dtype=np.int32)
            
            logger.info(f"\nFinal dataset:")
            logger.info(f"Features shape: {X.shape}")
            logger.info(f"Labels shape: {y.shape}")
            logger.info(f"Label distribution: {np.unique(y, return_counts=True)}")
            
            # Save processed video information
            output_dir = Path('outputs')
            output_dir.mkdir(exist_ok=True)
            
            with open(output_dir / 'processed_videos.txt', 'w') as f:
                for vid in processed_videos:
                    f.write(f"{vid}\n")
            
            return X, y
            
        except Exception as e:
            logger.error(f"\nError preparing dataset: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            raise
        
        def build_model(self, input_shape, num_classes):
            inputs = Input(shape=input_shape)
            
            x = LSTM(512, return_sequences=True)(inputs)
            x = LayerNormalization()(x)
            x = Dropout(0.2)(x)
            
            x = LSTM(256, return_sequences=True)(x)
            x = LayerNormalization()(x)
            x = Dropout(0.2)(x)
            
            x = LSTM(128)(x)
            x = LayerNormalization()(x)
            x = Dropout(0.2)(x)
            
            x1 = Dense(128, activation='relu')(x)
            x1 = Dropout(0.3)(x1)
            
            x2 = Dense(128, activation='relu')(x1)
            x2 = Dropout(0.2)(x2)
            
            # Concatenate features
            x = tf.keras.layers.Concatenate()([x1, x2])
            
            x = Dense(64, activation='relu')(x)
            outputs = Dense(num_classes, activation='softmax')(x)
            
            model = Model(inputs=inputs, outputs=outputs)
            
            # Using a schedule Learning rate here
            
            initial_learning_rate = 1e-3
            decay_steps = 1000
            decay_rate = 0.8
            learning_rate_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
                initial_learning_rate, decay_steps, decay_rate
            )
            
            optimizer = SGD(learning_rate=learning_rate_schedule)
            
            model.compile(
                optimizer=optimizer,
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy']
            )
            return model
        
        def train(self, video_dir, metadata_path, epochs=50, batch_size=16):
            X, y = self.prepare_data(video_dir, metadata_path)
            
            # Shuffle the data
            indices = np.arange(len(X))
            np.random.shuffle(indices)
            X = X[indices]
            y = y[indices]
            
            # Split the data
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=482, stratify=y
            )
            
            input_shape = (X.shape[1], X.shape[2])
            
            self.model = self.build_model(
                input_shape=input_shape,
                num_classes=len(np.unique(y))
            )
            
            output_dir = Path('outputs')
            output_dir.mkdir(exist_ok=True)
            
            # Enhanced callbacks
            callbacks = [
                EarlyStopping(
                    monitor='val_accuracy',
                    patience=15,
                    restore_best_weights=True,
                    verbose=1
                ),
                ModelCheckpoint(
                    output_dir / 'best_model.keras',
                    monitor='val_accuracy',
                    save_best_only=True,
                    verbose=1
                ),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss',
                    factor=0.2,
                    patience=5,
                    min_lr=1e-6,
                    verbose=1
                ),
                tf.keras.callbacks.CSVLogger(
                    output_dir / 'training_log.csv'
                )
            ]
            
            # Training the model
            history = self.model.fit(
                X_train, y_train,
                validation_split=0.2,
                epochs=epochs,
                batch_size=batch_size,
                callbacks=callbacks,
                shuffle=True,
                verbose=1
            )
            
            self.evaluate(X_test, y_test, history)
            
            
        def evaluate(self, X_test, y_test, history):
            
            output_dir = Path('outputs')
            output_dir.mkdir(exist_ok=True)
            
            y_pred = np.argmax(self.model.predict(X_test), axis=1)
            
            # Plotting
            plt.figure(figsize=(12, 4))
            
            plt.subplot(1, 2, 1)
            plt.plot(history.history['accuracy'], label='Train')
            plt.plot(history.history['val_accuracy'], label='Validation')
            plt.title('Model Accuracy')
            plt.xlabel('Epoch')
            plt.ylabel('Accuracy')
            plt.legend()
            
            plt.subplot(1, 2, 2)
            plt.plot(history.history['loss'], label='Train')
            plt.plot(history.history['val_loss'], label='Validation')
            plt.title('Model Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            
            plt.tight_layout()
            plt.savefig(output_dir / 'training_history.png')
            plt.close()
            
            # Plot confusion matrix
            plt.figure(figsize=(10, 8))
            cm = confusion_matrix(y_test, y_pred)
            cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
            
            sns.heatmap(
                cm_normalized,
                annot=cm,
                fmt='d',
                cmap='Blues',
                xticklabels=['Not Funny', 'Slightly Funny', 'Very Funny'],
                yticklabels=['Not Funny', 'Slightly Funny', 'Very Funny']
            )
            plt.title('Confusion Matrix\n(values: counts, colors: percentages)')
            plt.xlabel('Predicted')
            plt.ylabel('True')
            plt.savefig(output_dir / 'confusion_matrix.png', bbox_inches='tight')
            plt.close()
            
            # Save classification report
            report = classification_report(
                y_test,
                y_pred,
                target_names=['Not Funny', 'Slightly Funny', 'Very Funny'],
                digits=4
            )
            logger.info("\nClassification Report:\n%s", report)
            
            with open(output_dir / 'classification_report.txt', 'w') as f:
                f.write(report)
            
            # Save model summary
            with open(output_dir / 'model_summary.txt', 'w') as f:
                self.model.summary(print_fn=lambda x: f.write(x + '\n'))

In [ ]:
try:
    video_dir = '/Users/jeremiah/video-p-reaction-joketype-modular/data/videos'
    metadata_path = '/Users/jeremiah/video-p-reaction-joketype-modular/data/excel.xlsx'
    
    # Set random seeds for reproducibility
    np.random.seed(42)
    tf.random.set_seed(42)
    random.seed(42)
    
    # Configure GPU memory growth
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    
    logger.info(f"Video directory: {video_dir}")
    logger.info(f"Metadata path: {metadata_path}")
    
    classifier = VideoClassifier(max_frames=30, frame_size=(224, 224))
    classifier.train(
        video_dir=video_dir,
        metadata_path=metadata_path,
        epochs=50,
        batch_size=32
    )
except Exception as e:
    logger.error(f"Error in main: {str(e)}")
    import traceback
    logger.error(traceback.format_exc())

In [ ]:
# Advanced LSTM model with attention mechanism and residual connections.